# Post5G - RAN Split

## Prerequisites
* You have successfully followed the [Simple Scenario](./simple.ipynb) and [Adding component](./add_components.ipynb) tutorials.
* You have basic Python knowledge
* You have a good understanding of the 5G architecture
* You have a good understanding of the RAN split

## Experiment for this scenario

In this scenario we implement RAN splitting. We will consider the case of the
*CU - DU* split and the more advance case of *CU-CP/CU-UP - DU* split.

In our network on the one hand, the DU named `du1` is connected to the CU named
`cu`. On the other hand, the DU named `du2` is connected to the CU-CP named 
`cucp` and to the CU-UP named `cuup`. The UE `sim_ue1` is connected to `du1` and
the UE `sim_ue2` is connected to `du2` as illustrated in the figure below.

Each DU has its own Cell ID and PCI.

The 5G core is implemented with `Open5GS`, and the RAN is implemented with OAI.
The two UEs are implemented with `OAI` and the  radio channel is simulated with
rfsim.

![Split topology](figures/split.png)





## Metamodel and Deployement model definition

### Metamodel

The metamodel follows the same principle as in the other scenarios, except that
instead of defining a monolithic gNB, we define the split functions and how they
are connected.

We declare that the DU `du1` is connected to the CU `cu` by adding the `cu_cp`
and `cu_up` attributes to its `config` section. As `cu` implements at the same
time the CP and the UP, we reference it in both case.

On the contrary the DU `du2` is connected to the CU-CP `cucp` and to the CU-UP
`cuup`, we then explicitly reference them in the the `cu_cp` and `cu_up`
attributes of the DU. In addition, we have to specify that `cuup` has an E1
interface with the CU-CP `cucp`. This is why we have to add the `cu_cp`
attribute to the `config` section of `cuup`.

In addition, we specify a specific DU ID, CellID, and PCI to each DU for the
network to distinguish them.

In [1]:
from metamodel.models.metamodel import MetaModel

metamodel=MetaModel.from_yaml(
"""
ran:
    components:
      cu:
        type: cu

      cucp:
        type: cu_cp

      cuup:
        type: cu_up
        config:
          cu_cp: cucp

      du1:
        type: du
        config:
          gnbDuId: 1
          nrCellId: 1
          physicalCellId: 1
          cu_cp: cu
          cu_up: cu

      du2:
        type: du
        config:
          gnbDuId: 2
          nrCellId: 2
          physicalCellId: 2
          cu_cp: cucp
          cu_up: cuup
    
ues:
  sim_ue1:
    config:
      supi: '001010000000100'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - du1
  sim_ue2:
    config:
      supi: '001010000000101'
      subscriptions:
      - snssai: internet
        dnn: internet
      gnbs:
      - du2
""")

### Deployment model

Beside declaring each component constituting the split RAN, there is no
specific settings to add to the deployment model to implement the split.

In [2]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.from_yaml(
"""
ran:
  implementation: oai
  components:
    cu:
      type: cu
    cucp:
      type: cu_cp
    cuup:
      type: cu_up
    du1:
      type: du
    du2:
      type: du

ues:
  sim_ue1:
    implementation: oai
    type: containerized
  sim_ue2:
    implementation: oai
    type: containerized

nodes:
  node-core:
    provisioner: my-provisionner
    roles:
    - core
    - ric
  node-ran:
    provisioner: my-provisionner
    roles:
    - ran
  node-ues:
    provisioner: my-provisionner
    roles:
    - ue

provisioners:
  my-provisionner:
    type: bi
    params:
      infra: fr-sophia2-bi-vm1
      project: post5g-beta
      experiment: split_scenario
      create_experiment: true
      experiment_duration: 8h
      node_duration: 8h
"""
)



## Provisioning the infrastructure

We can provision the infrastructure as in the [Simple Scenario tutorial](./simple.ipynb).

In [3]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision(quiet=False)

k8s = K8SManagement(preceding=prov)
await k8s.deploy(quiet=True)

[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: No inventory was parsed, only implicit localhost is available
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'
[WARNING]: provided hosts list is empty, only localhost is available. Note that the implicit localhost does not match 'all'

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

PLAY [localhost] ***************************************************************

TASK [provisioning/bi/provision : Select project 'post5g-beta' for 'node-core' provisioning] ***

TASK [provisioning/bi/provision : Select project 'post5g-beta' f

At this stage, the infrastructure is fully ready to support our 5G network
deployment.

But before deploying the 5G network, let's archive the metamodel and deployment
model resulting from this deployment.

In [4]:
from pathlib import Path
DIR="state/split"
Path(DIR).mkdir(parents=True, exist_ok=True)

In [5]:
prov.metamodel.to_yaml_file(f"{DIR}/metamodel.prov.yaml")
prov.deployment_model.to_yaml_file(f"{DIR}/deployment_model.prov.yaml")

k8s.metamodel.to_yaml_file(f"{DIR}/metamodel.k8s.yaml")
k8s.deployment_model.to_yaml_file(f"{DIR}/deployment_model.k8s.yaml")

## Deploy the 5G Network

### Deploy the 5G core

Now that the infrastructure is fully provisioned, we first have to deploy the 5G
core.

In [7]:
from orchestrator.deployment import CoreManagement
core = CoreManagement(preceding=k8s)
await core.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'open5gs-pcf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-upf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-amf' chart] ******

TASK [helm/prepare : Create Helm build directory for 'open5gs-database' chart] ***

PLAY [node-co

### Deploy the 5G RAN

We can now deploy the 5G RAN and verify that the RAN split functions are
deployed properly.

In [8]:
prov = ProvisioningManagement(preceding=core)

print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "open5gs-core", "-o", "wide"],
    nodes=["node-core"],
    become=True, quiet=True))["node-core"])

NAME                            READY   STATUS    RESTARTS   AGE     IP              NODE        NOMINATED NODE   READINESS GATES
open5gs-amf-7b6dcfbfdf-m9lcp    1/1     Running   0          2m36s   10.244.134.7    node-core   <none>           <none>
open5gs-ausf-5778d7548b-2565d   1/1     Running   0          2m19s   10.244.134.8    node-core   <none>           <none>
open5gs-bsf-75cf9d7fb4-vx2jz    1/1     Running   0          31s     10.244.134.14   node-core   <none>           <none>
open5gs-database-0              1/1     Running   0          3m27s   10.244.134.5    node-core   <none>           <none>
open5gs-nrf-65d98494d7-7jst7    1/1     Running   0          2m58s   10.244.134.6    node-core   <none>           <none>
open5gs-nssf-84cb8d64f7-q9jpj   1/1     Running   0          49s     10.244.134.13   node-core   <none>           <none>
open5gs-pcf-6f444b66c5-6gjmh    1/1     Running   0          103s    10.244.134.10   node-core   <none>           <none>
open5gs-smf-5c7c8d4c4c-

In [9]:
from orchestrator.deployment import RANManagement
ran = RANManagement(preceding=core)
await ran.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cu' chart] ***********

TASK [helm/prepare : Create Helm build directory for 'oai-du1' chart] **********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cucp' chart] *********

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-cuup' chart] *********

TASK [helm/prepare : Create Helm build directory for 'oai-du2' chart] **********
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different inter

We can verify that all functions are deployed in the kubernetes cluster by
checking the pods deployed in the `oai-ran` namespace.

In [10]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ran", "-o", "wide"],
    nodes=[prov.deployment_model.params.k8s.master_node],
    become=True, quiet=True))[prov.deployment_model.params.k8s.master_node])

NAME                        READY   STATUS    RESTARTS   AGE   IP               NODE       NOMINATED NODE   READINESS GATES
oai-cu-7f4bd9b55b-p2r29     1/1     Running   0          55s   10.244.33.168    node-ran   <none>           <none>
oai-cucp-56b7476768-6xwlw   1/1     Running   0          55s   10.244.186.82    node-ran   <none>           <none>
oai-cuup-54f6484d49-l4k98   1/1     Running   0          38s   10.244.93.99     node-ran   <none>           <none>
oai-du1-66bc65b4f6-pzt2p    1/1     Running   0          27s   10.244.61.211    node-ran   <none>           <none>
oai-du2-5966588cb4-xh9rg    1/1     Running   0          27s   10.244.252.220   node-ran   <none>           <none>


## Deploy the UEs

Now that we have a fully functional core and RAN, we can deploy the UEs we have
defined in our metamodel.

In [11]:
from orchestrator.deployment import UEManagement
ues = UEManagement(preceding=ran)
await ues.deploy()


PLAY [node-core] ***************************************************************

PLAY [node-core] ***************************************************************

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue2' chart] ******

TASK [helm/prepare : Create Helm build directory for 'oai-sim-ue1' chart] ******
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed: [node-core]
[WARNING]: Host 'node-core' is using the discovered Python interpreter at '/usr/bin/python3.12', but future installation of another Python interpreter could cause a different interpreter to be discovered. See https://docs.ansible.com/ansible-core/2.21/reference_appendices/interpreter_discovery.html for more information.
changed

We can check that the two UEs have been deployed by listing the pods in the
`oai-ue` namespace.

In [12]:
print((await prov.run_node_command(
    command=["kubectl", "get", "pods", "-n", "oai-ue", "-o", "wide"],
    nodes=["node-ues"],
    become=True, quiet=True))["node-ues"])

NAME                           READY   STATUS    RESTARTS   AGE   IP             NODE       NOMINATED NODE   READINESS GATES
oai-sim-ue1-77fdccc764-hnv7s   1/1     Running   0          35s   10.244.198.2   node-ues   <none>           <none>
oai-sim-ue2-6644849f47-m6hmm   1/1     Running   0          34s   10.244.198.3   node-ues   <none>           <none>


We can also analyse the logs the the UEs and see that they both successuflly
registered to the network and established a PDU session.

In [13]:
print(await ues.get_logs(components=["sim_ue1"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

In [14]:
print(await ues.get_logs(components=["sim_ue2"], quiet=True))

CMDLINE: "/opt/oai-nr-ue/bin/nr-uesoftmodem" "-O" "/opt/oai-nr-ue/etc/nr-ue.conf" "-E" "--rfsim" 
[UTIL]   running in SA mode (no --phy-test, --do-ra, --nsa option present)
[UTIL]   threadCreate() for Tpool0_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool1_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool2_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool3_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool4_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool5_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool6_-1: creating thread with affinity ffffffff, priority 97
[UTIL]   threadCreate() for Tpool7_-1: creating thread with affinity ffffffff, priority 97
[OPT]    OPT disabled
[HW]     Version: Branch: develop-c8b78200bf7392b842f7e5b8485837531acaa3e4 Ab

To test that the UEs are well connected and that our network works as expected
we can run a ping from the UEs via their radio interace. In UERANSIM, the
interface is called `uesimtun0`.

To run commands directly on the nodes,

In [15]:
command = ["ping", "-I", "oaitun_ue1", "-c", 10, "-D", "8.8.8.8"]
pings = await ues.run_command(command=command, components=["sim_ue1", "sim_ue2"], quiet=True)

ping from `sim_ue1`:

In [16]:
print (pings["sim_ue1"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.2 oaitun_ue1: 56(84) bytes of data.
[1791790549.223335] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=57.2 ms
[1791790550.202455] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=35.1 ms
[1791790551.205260] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=37.5 ms
[1791790552.200653] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=32.3 ms
[1791790553.202176] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=32.4 ms
[1791790554.212751] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=42.4 ms
[1791790555.210401] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=39.0 ms
[1791790556.201795] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=29.4 ms
[1791790557.205392] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=31.5 ms
[1791790558.212839] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=38.4 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9008ms
rtt min/avg/max/mdev = 29.397/37.524/57.182/7.580 ms



ping from `sim_ue2`:

In [17]:
print (pings["sim_ue2"])

PING 8.8.8.8 (8.8.8.8) from 11.1.1.3 oaitun_ue1: 56(84) bytes of data.
[1791790549.357491] 64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=51.7 ms
[1791790550.334397] 64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=27.0 ms
[1791790551.340833] 64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=32.4 ms
[1791790552.341578] 64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=31.5 ms
[1791790553.346545] 64 bytes from 8.8.8.8: icmp_seq=5 ttl=114 time=34.9 ms
[1791790554.352839] 64 bytes from 8.8.8.8: icmp_seq=6 ttl=114 time=39.5 ms
[1791790555.350563] 64 bytes from 8.8.8.8: icmp_seq=7 ttl=114 time=36.2 ms
[1791790556.349423] 64 bytes from 8.8.8.8: icmp_seq=8 ttl=114 time=33.7 ms
[1791790557.350992] 64 bytes from 8.8.8.8: icmp_seq=9 ttl=114 time=33.5 ms
[1791790558.356787] 64 bytes from 8.8.8.8: icmp_seq=10 ttl=114 time=37.7 ms

--- 8.8.8.8 ping statistics ---
10 packets transmitted, 10 received, 0% packet loss, time 9013ms
rtt min/avg/max/mdev = 27.004/35.810/51.656/6.217 ms



Finally, we can archive the metamodel and deployment model in file for keeping
track of the experiment models.

In [19]:
ues.metamodel.to_yaml_file(f"{DIR}/metamodel.ues.yaml")
ues.deployment_model.to_yaml_file(f"{DIR}/deployment_model.ues.yaml")